# Robo-Greeno Hexapod — Body-Pose Control

**Problem A.** All six feet stay **planted** on the ground; only the **body** moves. The hexapod rises, lowers, pitches forward and back, then rolls — a smooth keyframed routine.

The trick: a foot that is fixed in the world is *not* fixed in the **body frame** when the body tilts. So each frame we take the six planted feet, re-express them in the moving body frame (`body_pose_targets`), and hand those six targets to the inverse kinematics. One height/pitch/roll command drives all 18 servos.

This is the Colab version of `HadasSigaron/demo_body_pose.py` — the same `config.py`, `hexapod_ik.py` and `hexapod_model.py`. On the desktop you'd run `python demo_body_pose.py` for the interactive 3D viewer; here we render an inline video.

**How to run it:** `Runtime → Run all`. A standard **CPU** Colab runtime is fine — no GPU needed.

## Setup

Install MuJoCo (the physics engine) and `mediapy` (to show videos inline).

In [ ]:
!pip install -q mujoco mediapy

In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"   # offscreen rendering backend (works on Colab)

import math
import numpy as np
import mujoco
import mediapy as media

print("MuJoCo", mujoco.__version__, "- ready")

## 1 · The robot, in three files

The exact source modules, written to the Colab filesystem so they `import` just like on the desktop:

- **`config.py`** — the *one config*: every length, angle and joint limit.
- **`hexapod_ik.py`** — closed-form inverse kinematics for one 3-DOF leg.
- **`hexapod_model.py`** — builds the single-robot MuJoCo model (MJCF) from the config.

In [ ]:
%%writefile config.py
"""
config.py  --  the ONE geometry file for the Robo-Greeno hexapod.

Everything about the robot's shape lives here. The MJCF model, the
inverse kinematics and the runner all read these numbers and nothing
else. When the real PhantomX hardware lands, you edit this file only
-- no other file needs to change. That is the whole point.

Units: metres and radians.  Frame: +X forward, +Y left, +Z up.
"""

import math

# --------------------------------------------------------------------
# Leg link lengths  (one 3-DOF leg: coxa -> femur -> tibia)
# Same 4 : 8 : 13 proportions as the Stage A leg explorer, in metres.
# Swap for real AX-12 dimensions when the hardware lands.
# --------------------------------------------------------------------
COXA  = 0.040   # L1  hip link  (horizontal swing link)
FEMUR = 0.080   # L2  thigh
TIBIA = 0.130   # L3  shin

FOOT_RADIUS = 0.012   # rounded foot tip -- also the ground contact radius

# --------------------------------------------------------------------
# Body
# --------------------------------------------------------------------
BODY_RADIUS = 0.100   # centre of body  ->  each coxa joint
BODY_HALF_H = 0.018   # half the trunk thickness
TRUNK_MASS  = 0.45    # kg  (small-scale learning model)

# --------------------------------------------------------------------
# The six legs.  Each is (name, mount angle).
# Mount angle: body XY-plane, 0 = forward, positive = CCW (toward +Y).
# --------------------------------------------------------------------
LEGS = [
    ("front_left",   math.radians(  45.0)),
    ("mid_left",     math.radians(  90.0)),
    ("back_left",    math.radians( 135.0)),
    ("back_right",   math.radians(-135.0)),
    ("mid_right",    math.radians( -90.0)),
    ("front_right",  math.radians( -45.0)),
]

# --------------------------------------------------------------------
# Alternating tripod gait groups (indices into LEGS).
# One tripod is on the ground while the other swings.
# --------------------------------------------------------------------
TRIPOD_A = [0, 2, 4]   # front_left, back_left, mid_right
TRIPOD_B = [1, 3, 5]   # mid_left,  back_right, front_right

# --------------------------------------------------------------------
# Joint travel limits  (radians)
# --------------------------------------------------------------------
COXA_RANGE  = (math.radians(-50.0), math.radians( 50.0))
FEMUR_RANGE = (math.radians(-90.0), math.radians(120.0))
TIBIA_RANGE = (math.radians(-170.0), math.radians( 20.0))

# --------------------------------------------------------------------
# Default standing stance
#   STANCE_RADIUS  -- foot distance from body centre, on the ground
#   STANCE_HEIGHT  -- height of the body centre above the ground
# --------------------------------------------------------------------
STANCE_RADIUS = 0.200
STANCE_HEIGHT = 0.075

# --------------------------------------------------------------------
# Tripod walk parameters (used by run.py --walk)
# --------------------------------------------------------------------
GAIT_PERIOD = 1.4     # seconds for one full A+B cycle
GAIT_STRIDE = 0.060   # metres a foot travels along +X per cycle
GAIT_LIFT   = 0.030   # metres a swinging foot lifts off the ground


def describe():
    """Print a short human summary of the configured robot."""
    span = 2.0 * STANCE_RADIUS
    leg_reach = COXA + FEMUR + TIBIA
    print("Robo-Greeno hexapod -- configured geometry")
    print(f"  legs            : {len(LEGS)}  x 3 DOF = {3 * len(LEGS)} joints")
    print(f"  link lengths    : coxa {COXA*100:.0f} cm | "
          f"femur {FEMUR*100:.0f} cm | tibia {TIBIA*100:.0f} cm")
    print(f"  max leg reach   : {leg_reach*100:.0f} cm")
    print(f"  stance width    : {span*100:.0f} cm  (foot to foot)")
    print(f"  body ride height: {STANCE_HEIGHT*100:.0f} cm")


if __name__ == "__main__":
    describe()


In [ ]:
%%writefile hexapod_ik.py
"""
hexapod_ik.py  --  closed-form kinematics for one 3-DOF hexapod leg.

This is the exact same maths as the Stage A leg explorer
(hexapod-leg-ik-explorer.html), packaged for the MuJoCo runner.
No iteration, no solver -- pure trigonometry students can derive on
paper and check by hand.

A leg has three joints:
  coxa   -- yaw, swings the whole leg left/right
  femur  -- pitch, lifts the leg in its own vertical plane
  tibia  -- pitch at the knee

Joint sign convention (matches the MJCF model):
  coxa  > 0  -> leg swings toward +Y
  femur > 0  -> leg lifts upward
  tibia      -> 0 is a straight leg, negative folds the foot under
"""

import math

import config as cfg


def leg_ik(x, y, z, L1=cfg.COXA, L2=cfg.FEMUR, L3=cfg.TIBIA):
    """Inverse kinematics: foot target -> three joint angles.

    x, y, z : desired foot position relative to the COXA joint, in
              that leg's own frame (+x points out along the leg).
    Returns (coxa, femur, tibia) in radians, or None if out of reach.
    """
    coxa = math.atan2(y, x)                 # top view: aim the leg
    r    = math.hypot(x, y)                 # horizontal run to the foot
    rho  = r - L1                           # reach past the coxa link
    D    = math.hypot(rho, z)               # femur joint -> foot

    # reach test: the femur+tibia 2-link arm must be able to span D
    if not (abs(L2 - L3) - 1e-9 <= D <= L2 + L3 + 1e-9):
        return None

    cos_knee = (L2 * L2 + L3 * L3 - D * D) / (2.0 * L2 * L3)
    knee     = math.acos(_clamp(cos_knee))

    cos_beta = (L2 * L2 + D * D - L3 * L3) / (2.0 * L2 * D)
    beta     = math.acos(_clamp(cos_beta))

    femur = math.atan2(z, rho) + beta       # knee-up solution
    tibia = knee - math.pi                  # 0 = straight leg
    return (coxa, femur, tibia)


def leg_fk(coxa, femur, tibia, L1=cfg.COXA, L2=cfg.FEMUR, L3=cfg.TIBIA):
    """Forward kinematics: three joint angles -> foot (x, y, z).
    Used to verify the IK by round-trip:  FK(IK(target)) == target."""
    pitch_t = femur + tibia                 # absolute pitch of the tibia
    rho_foot = L1 + L2 * math.cos(femur) + L3 * math.cos(pitch_t)
    z_foot = L2 * math.sin(femur) + L3 * math.sin(pitch_t)
    return (rho_foot * math.cos(coxa),
            rho_foot * math.sin(coxa),
            z_foot)


def body_target_to_leg(foot_body, mount_angle, body_radius=cfg.BODY_RADIUS):
    """Body frame -> leg frame.

    A foot target is naturally given in the body frame. Each leg's
    coxa joint sits on the body rim at its mount angle and the leg's
    own +x axis points radially outward. This rotates a body-frame
    target into the leg frame that leg_ik() expects."""
    fx, fy, fz = foot_body
    dx = fx - body_radius * math.cos(mount_angle)
    dy = fy - body_radius * math.sin(mount_angle)
    c, s = math.cos(mount_angle), math.sin(mount_angle)
    return (dx * c + dy * s,
            -dx * s + dy * c,
            fz)


def solve_all(foot_targets_body):
    """Solve every leg at once.

    foot_targets_body : 6 (x, y, z) foot targets in the body frame,
                        one per leg in config.LEGS order.
    Returns 6 (coxa, femur, tibia) tuples.
    Raises ValueError if any leg cannot reach its target."""
    angles = []
    for (name, mount), target in zip(cfg.LEGS, foot_targets_body):
        leg_xyz = body_target_to_leg(target, mount)
        sol = leg_ik(*leg_xyz)
        if sol is None:
            raise ValueError(f"leg '{name}' cannot reach {target}")
        angles.append(sol)
    return angles


def default_stance(stance_radius=None, stance_height=None):
    """Return 6 foot targets in the body frame for a neutral stand.

    The body frame sits at the trunk centre, so a foot resting on the
    ground is FOOT_RADIUS - STANCE_HEIGHT below it (the rounded foot
    tip touches the ground, its centre sits one radius higher)."""
    R = cfg.STANCE_RADIUS if stance_radius is None else stance_radius
    H = cfg.STANCE_HEIGHT if stance_height is None else stance_height
    foot_z = cfg.FOOT_RADIUS - H
    return [(R * math.cos(mount), R * math.sin(mount), foot_z)
            for name, mount in cfg.LEGS]


def _clamp(v, lo=-1.0, hi=1.0):
    return max(lo, min(hi, v))


if __name__ == "__main__":
    print("IK / FK round-trip on the default standing stance\n")
    worst = 0.0
    for (name, mount), target in zip(cfg.LEGS, default_stance()):
        leg_xyz = body_target_to_leg(target, mount)
        sol = leg_ik(*leg_xyz)
        err = math.dist(leg_xyz, leg_fk(*sol))
        worst = max(worst, err)
        deg = tuple(round(math.degrees(a), 1) for a in sol)
        print(f"  {name:12s}  coxa/femur/tibia = {deg}  err = {err:.2e} m")
    print(f"\nworst round-trip error: {worst:.2e} m  "
          f"({'PASS' if worst < 1e-9 else 'FAIL'})")


In [ ]:
%%writefile hexapod_model.py
"""
hexapod_model.py  --  build the MuJoCo model of the hexapod.

The whole robot is generated in code from config.py, so there is no
separate hand-edited XML to keep in sync. Call build_mjcf() to get the
MJCF (MuJoCo's XML) as a string, or run this file directly to write a
hexapod.xml you can inspect.

A self-contained PhantomX-class hexapod: a round trunk with six
identical 3-DOF legs, 18 hinge joints, one position servo per joint,
and a ground plane. No external mesh or URDF download needed.
"""

import math

import config as cfg


def _leg(name, mount):
    """Return the MJCF body block for one leg: coxa -> femur -> tibia."""
    rx = cfg.BODY_RADIUS * math.cos(mount)
    ry = cfg.BODY_RADIUS * math.sin(mount)
    c0, c1 = cfg.COXA_RANGE
    f0, f1 = cfg.FEMUR_RANGE
    t0, t1 = cfg.TIBIA_RANGE
    L1, L2, L3 = cfg.COXA, cfg.FEMUR, cfg.TIBIA
    rf = cfg.FOOT_RADIUS
    return f"""
      <body name="{name}_coxa" pos="{rx:.6f} {ry:.6f} 0" euler="0 0 {mount:.6f}">
        <joint name="{name}_coxa" axis="0 0 1" range="{c0:.6f} {c1:.6f}"/>
        <geom type="capsule" fromto="0 0 0 {L1:.6f} 0 0" size="0.012" rgba="0.60 0.58 0.54 1"/>
        <body name="{name}_femur" pos="{L1:.6f} 0 0">
          <joint name="{name}_femur" axis="0 -1 0" range="{f0:.6f} {f1:.6f}"/>
          <geom type="capsule" fromto="0 0 0 {L2:.6f} 0 0" size="0.010" rgba="0.11 0.62 0.46 1"/>
          <body name="{name}_tibia" pos="{L2:.6f} 0 0">
            <joint name="{name}_tibia" axis="0 -1 0" range="{t0:.6f} {t1:.6f}"/>
            <geom type="capsule" fromto="0 0 0 {L3:.6f} 0 0" size="0.008" rgba="0.18 0.49 0.85 1"/>
            <geom name="{name}_foot" type="sphere" pos="{L3:.6f} 0 0" size="{rf:.6f}" rgba="0.85 0.35 0.19 1"/>
            <site name="{name}_foot" pos="{L3:.6f} 0 0" size="0.006"/>
          </body>
        </body>
      </body>"""


def _actuators():
    rows = []
    for name, _ in cfg.LEGS:
        for joint, rng in (("coxa", cfg.COXA_RANGE),
                           ("femur", cfg.FEMUR_RANGE),
                           ("tibia", cfg.TIBIA_RANGE)):
            kp = 18.0 if joint == "coxa" else 30.0
            rows.append(
                f'    <position name="{name}_{joint}" joint="{name}_{joint}" '
                f'kp="{kp}" ctrlrange="{rng[0]:.6f} {rng[1]:.6f}"/>')
    return "\n".join(rows)


def build_mjcf():
    """Return the complete MuJoCo model as an MJCF (XML) string."""
    legs = "".join(_leg(name, mount) for name, mount in cfg.LEGS)
    spawn_z = cfg.STANCE_HEIGHT
    return f"""<mujoco model="robo_greeno_hexapod">
  <compiler angle="radian" autolimits="true"/>
  <option timestep="0.002" integrator="implicitfast" gravity="0 0 -9.81"/>

  <default>
    <joint damping="0.14" armature="0.012"/>
    <geom friction="1.1 0.06 0.01" density="700"/>
  </default>

  <visual>
    <headlight diffuse="0.5 0.5 0.5" ambient="0.4 0.4 0.4"/>
    <rgba haze="0.95 0.95 0.93 1"/>
  </visual>

  <worldbody>
    <light pos="0 0 1.4" dir="0 0 -1" diffuse="0.7 0.7 0.7"/>
    <geom name="ground" type="plane" size="3 3 0.1" rgba="0.92 0.91 0.87 1"/>

    <body name="trunk" pos="0 0 {spawn_z:.4f}">
      <freejoint name="trunk"/>
      <geom name="trunk" type="cylinder" size="{cfg.BODY_RADIUS:.4f} {cfg.BODY_HALF_H:.4f}"
            mass="{cfg.TRUNK_MASS}" rgba="0.36 0.35 0.33 1"/>
      <site name="trunk_center" pos="0 0 0" size="0.01"/>{legs}
    </body>
  </worldbody>

  <actuator>
{_actuators()}
  </actuator>
</mujoco>
"""


def save(path="hexapod.xml"):
    """Write the generated MJCF to a file and return the path."""
    with open(path, "w") as fh:
        fh.write(build_mjcf())
    return path


if __name__ == "__main__":
    p = save()
    print(f"wrote {p}")
    print(f"  {3 * len(cfg.LEGS)} joints, {3 * len(cfg.LEGS)} position servos, "
          f"{len(cfg.LEGS)} legs")


## 2 · The body-pose demo

This is `demo_body_pose.py`, written out and imported. The two functions that do the work:

- `world_feet()` — the six foot positions on the ground (they never move).
- `body_pose_targets(height, pitch, roll)` — re-expresses those planted feet in the body frame after the body has been raised by `height` and rotated by `pitch`/`roll`. That is the inverse body rotation applied to each foot.

`pose_at(t)` walks a keyframe table (neutral → up → down → pitch fwd → pitch back → roll → neutral) with smooth-step blending, so the motion eases between poses.

In [ ]:
%%writefile demo_body_pose.py
"""
demo_body_pose.py  --  Robo-Greeno hexapod demo: body-pose control.

All six feet stay planted on the ground; only the BODY moves. The
robot rises, lowers, pitches forward and back, then rolls -- a smooth
keyframed routine. Every frame is just six foot targets (the planted
feet, re-expressed in the moving body frame) fed through the inverse
kinematics. No gait, no walking -- only posing.

This is Problem A (body-pose control): one solver drives the whole
robot from a single body height/pitch/roll command.

Run it
------
  pip install mujoco
  python demo_body_pose.py          # open the 3D viewer
  python demo_body_pose.py --check  # headless self-test, no display
"""

import argparse
import math
import sys
import time
import numpy as np

import config as cfg
import hexapod_ik as ik
import hexapod_model as model


# --------------------------------------------------------------------
# MuJoCo plumbing -- builds the robot and lets us pose it
# --------------------------------------------------------------------
def make_sim():
    import mujoco
    m = mujoco.MjModel.from_xml_string(model.build_mjcf())
    return mujoco, m, mujoco.MjData(m)


def _aid(mj, m, name):
    return mj.mj_name2id(m, mj.mjtObj.mjOBJ_ACTUATOR, name)


def _jadr(mj, m, name):
    return m.jnt_qposadr[mj.mj_name2id(m, mj.mjtObj.mjOBJ_JOINT, name)]


def init_stance(mj, m, d):
    """Start the robot already standing so it does not snap on spawn."""
    mj.mj_resetData(m, d)
    t = _jadr(mj, m, "trunk")
    d.qpos[t:t + 7] = [0, 0, cfg.STANCE_HEIGHT, 1, 0, 0, 0]
    for (name, mount), tgt in zip(cfg.LEGS, ik.default_stance()):
        coxa, femur, tibia = ik.leg_ik(*ik.body_target_to_leg(tgt, mount))
        for joint, val in (("coxa", coxa), ("femur", femur), ("tibia", tibia)):
            d.qpos[_jadr(mj, m, f"{name}_{joint}")] = val
    mj.mj_forward(m, d)


def command(mj, m, d, foot_targets):
    """Solve the IK for six foot targets and write the eighteen servos."""
    for (name, _), (coxa, femur, tibia) in zip(cfg.LEGS, ik.solve_all(foot_targets)):
        d.ctrl[_aid(mj, m, f"{name}_coxa")] = coxa
        d.ctrl[_aid(mj, m, f"{name}_femur")] = femur
        d.ctrl[_aid(mj, m, f"{name}_tibia")] = tibia

def world_feet():
    R = cfg.STANCE_RADIUS

    return [
        (R * math.cos(mount),
         R * math.sin(mount),
        0.0)
        for name, mount in cfg.LEGS
    ]

def body_pose_targets(height, pitch, roll):
    """Fixed feet in the world, moving body."""

    cp = math.cos(pitch)
    sp = -math.sin(pitch)

    cr = math.cos(roll)
    sr = math.sin(roll)

    targets = []

    for px, py, pz in world_feet():

        # p - body_centre
        x = px
        y = py
        z = pz - height 

        # R_body^T = R_pitch^T * R_roll^T

        # inverse roll (about X)
        x1 = x
        y1 = y * cr + z * sr
        z1 = -y * sr + z * cr

        # inverse pitch (about Y)
        x2 = x1 * cp - z1 * sp
        y2 = y1
        z2 = x1 * sp + z1 * cp

        targets.append((x2, y2, z2))

    return targets


# --------------------------------------------------------------------
# The demo  --  this is the part you own
# --------------------------------------------------------------------

_KEYS = [
    (0.0,  cfg.STANCE_HEIGHT,        0.0,               0.0,              "neutral"),
    (3.0,  cfg.STANCE_HEIGHT + 0.02, 0.0,               0.0,              "up"),
    (6.0,  cfg.STANCE_HEIGHT - 0.02, 0.0,               0.0,              "down"),
    (9.0,  cfg.STANCE_HEIGHT,        math.radians(4),   0.0,              "pitch forward"),
    (12.0, cfg.STANCE_HEIGHT,        math.radians(-8),  0.0,              "pitch back"),
    (15.0, cfg.STANCE_HEIGHT,        0.0,               math.radians(8),  "roll right"),
    (18.0, cfg.STANCE_HEIGHT,        0.0,               0.0,              "neutral"),
]


def smoothstep(u):
    u = max(0.0, min(1.0, u))
    return u * u * (3.0 - 2.0 * u)


def lerp(a, b, s):
    return a + (b - a) * s

def pose_at(t):
    t = t % 18.0

    for i in range(len(_KEYS) - 1):
        t0, h0, p0, r0, _ = _KEYS[i]
        t1, h1, p1, r1, name = _KEYS[i + 1]

        if t0 <= t <= t1:
            s = smoothstep((t - t0) / (t1 - t0))

            height = lerp(h0, h1, s)
            pitch = lerp(p0, p1, s)
            roll = lerp(r0, r1, s)

            return height, pitch, roll, name

    return cfg.STANCE_HEIGHT, 0.0, 0.0, "neutral"

# --------------------------------------------------------------------
# Viewer
# --------------------------------------------------------------------
def view():
    import mujoco
    import mujoco.viewer
    mj, m, d = make_sim()
    init_stance(mj, m, d)
    print("body-pose demo  --  drag to orbit, close the window to quit")
    label = ""
    with mujoco.viewer.launch_passive(m, d) as v:
        start = time.time()
        while v.is_running():
            t = time.time() - start
            h, pitch, roll, name = pose_at(t)
            targets = body_pose_targets(h, pitch, roll)
            if name != label:
                print(f"  [{t:5.1f}s]  {name}")
                label = name
            command(mj, m, d, targets)
            mj.mj_step(m, d)
            v.sync()
            wait = m.opt.timestep - (time.time() - start - t)
            if wait > 0:
                time.sleep(wait)


# --------------------------------------------------------------------
# Headless self-test
# --------------------------------------------------------------------
def check():
    print("body-pose demo  --  self-test\n")
    mj, m, d = make_sim()
    print(f"[1] model loaded: {m.nu} servos, {m.nbody} bodies")

    print("[2] every pose in the routine is reachable")
    bad = 0
    for k in range(72):                       # 18 s, every 0.25 s
        h, pitch, roll, _ = pose_at(k * 0.25)
        targets = body_pose_targets(h, pitch, roll)
        try:
            ik.solve_all(targets)
        except ValueError:
            bad += 1
    poses_ok = bad == 0
    print(f"    unreachable poses: {bad}  -> {'PASS' if poses_ok else 'FAIL'}")

    print("[3] the robot holds itself up through the whole routine")
    init_stance(mj, m, d)
    low = 1.0
    for _ in range(9000):                     # 18 s
        h, pitch, roll, _ = pose_at(d.time)
        targets = body_pose_targets(h, pitch, roll)
        command(mj, m, d, targets)
        mj.mj_step(m, d)
        low = min(low, float(d.qpos[_jadr(mj, m, "trunk") + 2]))
    upright = low > 0.035
    print(f"    lowest ride height: {low*100:.1f} cm  "
          f"-> {'PASS' if upright else 'FAIL'}")

    ok = poses_ok and upright
    print(f"\n{'ALL CHECKS PASSED' if ok else 'SOME CHECKS FAILED'}")
    return 0 if ok else 1


def main():
    ap = argparse.ArgumentParser(description="hexapod body-pose demo")
    ap.add_argument("--check", action="store_true", help="headless self-test")
    args = ap.parse_args()
    if args.check:
        return check()
    try:
        view()
    except ImportError:
        print("MuJoCo is not installed.  Run:  pip install mujoco")
        return 1
    except Exception as exc:
        print(f"could not open the viewer ({exc}).")
        print("Try the self-test instead:  python demo_pose_wave.py --check")
        return 1
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
import config as cfg
import hexapod_ik as ik
import hexapod_model as model
import demo_body_pose as demo

# the demo's own headless model builder returns (mujoco, model, data)
mj, m, d = demo.make_sim()
print(f"model: {m.nu} servos, {m.nbody} bodies")
print("pose schedule:", [(t, name) for (t, _, _, _, name) in demo._KEYS])

## 3 · Render it as a video

On the desktop, `demo.view()` opens an interactive window. Colab has no window, so we do the same thing **off-screen**: step the physics through the 18-second routine and capture a frame every few steps with a fixed camera that frames the robot. The feet stay put; watch the body rise, dip, tip and roll.

In [ ]:
def render_body_pose(seconds=18, fps=30, width=640, height=480):
    """Run the body-pose routine off-screen and return video frames."""
    mj, m, d = demo.make_sim()
    demo.init_stance(mj, m, d)

    renderer = mujoco.Renderer(m, height=height, width=width)
    cam = mujoco.MjvCamera()
    cam.azimuth = 130.0
    cam.elevation = -18.0
    cam.distance = 0.95
    cam.lookat[:] = [0.0, 0.0, 0.04]

    frames, labels = [], []
    frame_every = max(1, round((1.0 / fps) / m.opt.timestep))
    n_steps = int(seconds / m.opt.timestep)

    for k in range(n_steps):
        h, pitch, roll, name = demo.pose_at(d.time)
        demo.command(mj, m, d, demo.body_pose_targets(h, pitch, roll))
        mujoco.mj_step(m, d)
        if k % frame_every == 0:
            renderer.update_scene(d, camera=cam)
            frames.append(renderer.render())
            labels.append(name)
    return frames, labels

### The result

The feet never move; the body rises, lowers, pitches and rolls. Each phase is labelled below the video.

In [ ]:
frames, labels = render_body_pose(seconds=18, fps=30)
media.show_video(frames, fps=30)

## 4 · The numbers (self-test)

The same headless self-test that runs on the desktop with `python demo_body_pose.py --check`: it confirms every pose in the routine is reachable by the IK, and that the robot holds itself up through the whole 18-second routine.

In [ ]:
demo.check()

## Make it yours

Things to try by editing the cells above and re-running:

- **New poses** — add rows to `_KEYS` in `demo_body_pose.py` (re-run its `%%writefile` cell): a deeper crouch, a bigger roll, a twist.
- **Bigger motion** — increase the pitch/roll angles. If a pose becomes unreachable, the self-test's step [2] will flag it.
- **Camera** — change `cam.azimuth` / `cam.elevation` / `cam.distance` in `render_body_pose` to view the tilt from a different angle.

The desktop viewer (`python demo_body_pose.py`) and this notebook run the *same* kinematics — this is Problem A, body-pose control.